# T10 (a) — KEJ 적응: small b0 → 학습 → 파일 단위 평가

평가는 **02-04 녹음 전체**를 롱폼으로 돌려 라벨 전체와 비교한다(문장 분절 불필요).
학습은 02-03에서 자동 분절된 18쌍(2.4분) + 개발 5쌍. **인식기가 문장 앞뒤를 맞힌 부분집합**이라는 한계를 결과와 함께 적는다.

## 드라이브에 추가로 올릴 것 (`MyDrive/t10_highcer/` 안에)

`segments/` 폴더, `t10_pairs_manifest.json`, `b1_train.py`, `eval_longform.py`, `build_t10_pairs.py`

**런타임 T4.** 전체 30~40분 예상.

## 1. 설치 — `torchao 없음`이 찍혀야 한다

In [ ]:
!pip -q install peft rapidfuzz ; pip -q uninstall -y torchao ; python -c "import torch, peft, importlib.util as u; print('GPU', torch.cuda.is_available(), 'torchao 없음' if u.find_spec('torchao') is None else 'TORCHAO 남아있음 - 런타임 재시작')"


## 2. 드라이브

In [ ]:
from google.colab import drive; drive.mount('/content/drive')


## 3. small b0 — 평가 파일 기준값 (KEJ, DTH 02-04, 각 2~5분)

DTH가 small 기준으로 고CER인지 여기서 처음 판단한다.

In [ ]:
!cd "/content/drive/MyDrive/t10_highcer" && python eval_longform.py --id ID-01-13-N-KEJ-02-04-F-36-KK --wavdir . && python eval_longform.py --id ID-01-13-N-DTH-02-04-M-85-KK --wavdir .


## 4. 학습 (KEJ, 18쌍)

`RUNAWAY` 줄과 선택된 epoch를 확인한다. test 4문장 숫자는 스모크일 뿐이다.

In [ ]:
!cd "/content/drive/MyDrive/t10_highcer" && python b1_train.py --speaker KEJ --manifest t10_pairs_manifest.json --segdir segments --out results


## 5. b1 — 같은 경로로 파일 단위 평가

In [ ]:
!cd "/content/drive/MyDrive/t10_highcer" && python eval_longform.py --id ID-01-13-N-KEJ-02-04-F-36-KK --wavdir . --adapter results/KEJ_nall_s0/adapter


## 끝나면

`t10_highcer/results/` 를 통째로 내려받아 로컬 `experiments/t10_highcer/results/` 에 덮어쓴다.

| 증상 | 조치 |
| --- | --- |
| `missing ... wav` | wav 4개가 `t10_highcer/` 바로 안에 있는지 |
| `speaker KEJ not in manifest` | `t10_pairs_manifest.json` 업로드 확인 |
| `No such file ... segments` | `segments/` 폴더(27개 wav) 업로드 확인 |